In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df= pd.read_csv("powerconsumption.csv"),encoding="utf-8-sig")
print(df.shape)
df.head()

SyntaxError: unmatched ')' (1308472547.py, line 6)

In [ ]:
df= pd.read_csv("powerconsumption.csv",encoding="utf-8-sig")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("powerconsumption.csv", encoding="utf-8-sig")
print(df.shape)
df.head()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("powerconsumption.csv", encoding="utf-8-sig")
print(df.shape)
df.head()

In [ ]:
df.info()
print("Missing values:\n", df.isna().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate timestamps:", df["Datetime"].duplicated().sum())
df.describe().round(2)

In [ ]:
df["Datetime"] = pd.to_datetime(df["Datetime"], format="%m/%d/%Y %H:%M")
print(df["Datetime"].min(), "to", df["Datetime"].max())
gaps = df["Datetime"].diff().dropna()
print(gaps.value_counts())

## Data-quality notes
- No missing values and no duplicate rows or timestamps.
- Datetime was stored as text; converted to datetime format.
- Recording interval is consistent (10 minutes). Data runs 1 Jan to 30 Dec 2017 (31 Dec is absent).
- WindSpeed is unusual: about 58% of readings are below 0.2 and about 39% are near 5. It behaves like two levels, so treat wind results with caution.
- Zone 3 has 1,191 unusually high values and Zone 2 has 7. They are real demand peaks (see Task 2), so they are kept.
- Units are not given in the brief.

In [ ]:
df = df.drop_duplicates(subset="Datetime").reset_index(drop=True)
print("Missing values left:", df.isna().sum().sum())

df["Date"] = df["Datetime"].dt.normalize()
df["Year"] = df["Datetime"].dt.year
df["Month"] = df["Datetime"].dt.month
df["Month_Name"] = df["Datetime"].dt.month_name()
df["Day"] = df["Datetime"].dt.day
df["Hour"] = df["Datetime"].dt.hour
df["DayOfWeek"] = df["Datetime"].dt.day_name()
df["Is_Weekend"] = df["Datetime"].dt.dayofweek >= 5

zones = ["PowerConsumption_Zone1", "PowerConsumption_Zone2", "PowerConsumption_Zone3"]
df["Total_Consumption"] = df[zones].sum(axis=1)
df.head()

In [ ]:
for col in zones:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    flag = (df[col] < q1 - 1.5 * iqr) | (df[col] > q3 + 1.5 * iqr)
    df["Outlier_" + col[-5:]] = flag
    print(col, "unusual values:", flag.sum())

print(df[df["Outlier_Zone3"]]["Month_Name"].value_counts())

In [ ]:
df.to_csv("cleaned_powerconsumption.csv", index=False)
print(df.shape)

In [ ]:
cols = zones + ["Total_Consumption"]
summary = df[cols].agg(["mean", "median", "std", "min", "max"]).T.round(0)
summary["Share_%"] = (df[zones].mean() / df[zones].mean().sum() * 100).round(1)
summary

In [ ]:
daily = df.groupby("Date")[cols].mean()
print("Highest reading:", df.loc[df["Total_Consumption"].idxmax(), ["Datetime", "Total_Consumption"]].tolist())
print("Lowest reading:", df.loc[df["Total_Consumption"].idxmin(), ["Datetime", "Total_Consumption"]].tolist())
print("Highest days:\n", daily["Total_Consumption"].nlargest(3).round(0))
print("Lowest days:\n", daily["Total_Consumption"].nsmallest(3).round(0))

In [ ]:
hourly = df.groupby("Hour")[cols].mean().round(0)
print("Peak hour:", hourly["Total_Consumption"].idxmax(), "| Lowest hour:", hourly["Total_Consumption"].idxmin())
hourly

In [ ]:
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday = df.groupby("DayOfWeek")[cols].mean().reindex(order).round(0)
print(weekday)
print("\nWeekend vs weekday (% change):")
print((df.groupby("Is_Weekend")[cols].mean().pct_change().iloc[-1] * 100).round(1))

In [ ]:
months = ["January", "February", "March", "April", "May", "June", "July",
          "August", "September", "October", "November", "December"]
monthly = df.groupby("Month_Name")[cols].mean().reindex(months).round(0)
monthly

In [ ]:
print("Variability (std / mean):\n", (df[zones].std() / df[zones].mean()).round(2))
print("\nPeak month / lowest month:\n", (monthly[zones].max() / monthly[zones].min()).round(2))
print("\nCorrelation between zones:\n", df[zones].corr().round(2))

In [ ]:
env = ["Temperature", "Humidity", "WindSpeed", "GeneralDiffuseFlows", "DiffuseFlows"]
corr = df[env + cols].corr().loc[env, cols].round(2)
corr

In [ ]:
plt.figure(figsize=(8, 4))
sns.heatmap(corr, annot=True, cmap="RdBu_r", center=0, vmin=-1, vmax=1)
plt.title("Environmental factors vs consumption (correlation)")
plt.tight_layout()
plt.show()

In [ ]:
daily = df.groupby("Date")[env + cols].mean()
print(daily.corr().loc[env, "Total_Consumption"].round(2))
slope = np.polyfit(daily["Temperature"], daily["Total_Consumption"], 1)[0]
print("Change in total consumption per +1°C (daily):", round(slope))
print("Temperature vs Humidity correlation:", round(df["Temperature"].corr(df["Humidity"]), 2))

In [ ]:
df["Temp_Band"] = pd.cut(df["Temperature"], bins=[0, 10, 15, 20, 25, 30, 45],
                         labels=["≤10", "10-15", "15-20", "20-25", "25-30", ">30"])
bands = df.groupby("Temp_Band", observed=True)[cols].mean().round(0)
bands

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, var in zip(axes, ["Temperature", "Humidity", "WindSpeed"]):
    sns.regplot(data=daily, x=var, y="Total_Consumption", ax=ax,
                scatter_kws={"alpha": 0.5, "s": 15})
    r = daily[var].corr(daily["Total_Consumption"])
    ax.set_title(f"{var} (r = {r:.2f})")
plt.tight_layout()
plt.show()

## Task 4: Environmental relationships
- Temperature is the strongest factor: r ≈ 0.49 per reading and ≈ 0.73 on daily averages. Each +1°C goes with roughly +1,170 in total consumption. Readings above 30°C average about 95,500 vs about 55,700 at 10°C or below.
- Humidity is negatively related (r ≈ -0.30), but humidity and temperature move in opposite directions (r ≈ -0.46), so much of this overlaps with temperature.
- Wind speed and diffuse flows are weak at reading level. The wind data is two-level (see data-quality notes), so it should not be relied on.
- These are associations, not proof of causation. Season and time of day change temperature and consumption together.

In [ ]:
import os
os.makedirs("figures", exist_ok=True)
sns.set_theme(style="whitegrid")
colors = {"Zone 1": "#2E6F9E", "Zone 2": "#E08A1E", "Zone 3": "#3A9D78"}
labels = {"PowerConsumption_Zone1": "Zone 1",
          "PowerConsumption_Zone2": "Zone 2",
          "PowerConsumption_Zone3": "Zone 3"}

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
for col in zones:
    name = labels[col]
    ax.plot(daily.index, daily[col], color=colors[name], alpha=0.25, linewidth=0.8)
    ax.plot(daily.index, daily[col].rolling(7, center=True).mean(),
            color=colors[name], linewidth=2, label=name)
ax.set_title("Consumption peaks in July-August in all three zones")
ax.set_ylabel("Mean consumption per 10-min reading")
ax.legend()
plt.tight_layout()
plt.savefig("figures/01_trend_over_time.png", dpi=150)
plt.show()

In [ ]:
means = df[zones].mean()
names = [labels[c] for c in zones]
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(names, means.values, color=[colors[n] for n in names])
ax.bar_label(bars, labels=[f"{v:,.0f}\n({v / means.sum():.0%})" for v in means.values])
ax.set_ylim(0, means.max() * 1.2)
ax.set_title("Zone 1 accounts for about 45% of consumption")
ax.set_ylabel("Mean consumption per 10-min reading")
plt.tight_layout()
plt.savefig("figures/02_average_by_zone.png", dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
for col in zones:
    ax.plot(hourly.index, hourly[col], marker="o", markersize=3,
            color=colors[labels[col]], label=labels[col])
ax.axvspan(18, 21, color="grey", alpha=0.15)
ax.set_xticks(range(0, 24, 2))
ax.set_xlabel("Hour of day")
ax.set_ylabel("Mean consumption")
ax.set_title("Demand peaks at 20:00, about twice the 06:00 low")
ax.legend()
plt.tight_layout()
plt.savefig("figures/03_hourly_pattern.png", dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
for col in zones:
    ax.plot(range(1, 13), monthly[col].values, marker="o",
            color=colors[labels[col]], label=labels[col])
ax.set_xticks(range(1, 13))
ax.set_xticklabels([m[:3] for m in months])
ax.set_ylabel("Mean consumption")
ax.set_title("Zone 3 swings most between summer and winter")
ax.legend()
plt.tight_layout()
plt.savefig("figures/04_monthly_pattern.png", dpi=150)
plt.show()

In [ ]:
dev = (weekday[zones] / weekday[zones].mean() - 1) * 100
dev = dev.rename(columns=labels).rename_axis("DayOfWeek").reset_index()
dev = dev.melt(id_vars="DayOfWeek", var_name="Zone", value_name="Pct")
fig, ax = plt.subplots(figsize=(10, 4.5))
sns.barplot(data=dev, x="DayOfWeek", y="Pct", hue="Zone", palette=colors, ax=ax)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("")
ax.set_ylabel("% difference from the zone's weekly average")
ax.set_title("Weekends are lower in Zones 1 and 2, not in Zone 3")
plt.tight_layout()
plt.savefig("figures/05_day_of_week.png", dpi=150)
plt.show()

In [ ]:
pivot = df.pivot_table(index="Hour", columns="Month", values="Total_Consumption", aggfunc="mean")
plt.figure(figsize=(10, 6))
sns.heatmap(pivot, cmap="YlOrRd", xticklabels=[m[:3] for m in months],
            cbar_kws={"label": "Mean total consumption"})
plt.title("Highest demand is on summer evenings")
plt.tight_layout()
plt.savefig("figures/06_hour_month_heatmap.png", dpi=150)
plt.show()

In [ ]:
long = df[zones].rename(columns=labels).melt(var_name="Zone", value_name="Consumption")
fig, ax = plt.subplots(figsize=(7, 4.5))
sns.boxplot(data=long, x="Zone", y="Consumption", hue="Zone",
            palette=colors, legend=False, ax=ax)
ax.set_title("Zone 3 is the most variable, with many high peaks")
plt.tight_layout()
plt.savefig("figures/07_zone_comparison.png", dpi=150)
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
sns.heatmap(corr, annot=True, cmap="RdBu_r", center=0, vmin=-1, vmax=1)
plt.title("Environmental factors vs consumption (correlation)")
plt.tight_layout()
plt.savefig("figures/08_correlation_heatmap.png", dpi=150)
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, var in zip(axes, ["Temperature", "Humidity", "WindSpeed"]):
    sns.regplot(data=daily, x=var, y="Total_Consumption", ax=ax,
                scatter_kws={"alpha": 0.5, "s": 15})
    r = daily[var].corr(daily["Total_Consumption"])
    ax.set_title(f"{var} (r = {r:.2f})")
plt.tight_layout()
plt.savefig("figures/09_environment_scatter.png", dpi=150)
plt.show()

In [ ]:
import plotly.express as px

long_daily = daily[zones].rename(columns=labels).reset_index()
long_daily = long_daily.melt(id_vars="Date", var_name="Zone", value_name="Mean consumption")
fig = px.line(long_daily, x="Date", y="Mean consumption", color="Zone",
              color_discrete_map=colors,
              title="Daily mean consumption by zone (interactive)")
fig.write_html("figures/interactive_daily_trend.html")
fig.show()

In [ ]:
ig = px.imshow(pivot, aspect="auto", color_continuous_scale="YlOrRd",
                labels=dict(x="Month", y="Hour", color="Mean total"),
                title="Mean total consumption by hour and month (interactive)")
fig.write_html("figures/interactive_hour_month_heatmap.html")
fig.show()

In [ ]:
d = daily.reset_index()
d["Month"] = d["Date"].dt.month
fig = px.scatter(d, x="Temperature", y="Total_Consumption", color="Month",
                 hover_data=["Date"],
                 title="Daily temperature vs total consumption (interactive)")
fig.write_html("figures/interactive_temperature_scatter.html")
fig.show()

## Task 6: Business Insights and Recommendations

### Key findings
1. *Strong evening peak.* Consumption peaks at 20:00 (about 98,000 per reading across all zones), almost twice the 06:00 low (about 50,200). The 18:00–22:00 window is 17% of the day but carries about 22% of consumption.
2. *Summer surge.* July averages about 88,200 versus about 63,800 in December (+38%). The highest reading (about 134,200) was on 24 July at 20:10, and the highest day was 25 July.
3. *Zone 3 is the most seasonal and volatile.* Its July average is 2.55 times its December average, and its variability (0.37) is well above Zone 1 (0.22) and Zone 2 (0.25). All 1,191 of its unusually high readings fall in July and August, mostly in the evening.
4. *Zones do not behave alike.* Zone 1 is the largest (about 45% of consumption). Zone 2 stays high in winter (about 23,700 in December versus about 24,100 in July) and is lowest in April, so it does not follow a summer-only pattern.
5. *Temperature is the strongest environmental factor.* Correlation is about 0.49 per reading and 0.73 on daily averages. Each +1°C goes with roughly +1,170 in total consumption, and readings above 30°C average about 95,500 versus about 55,700 at 10°C or below.
6. *Weekends are lower in some zones.* Total consumption is about 3.8% lower at weekends: Zone 2 about 8.2% lower, Zone 1 about 3.5% lower, and Zone 3 about 1.2% higher.

### What this means for an energy management organisation
- Capacity needs are driven by short evening peaks and by hot summer days, not by average demand.
- Zone 3 carries the most seasonal risk, so one zone-wide plan will not fit all three zones.
- Temperature forecasts are a useful early signal of high-demand days.

### Recommendations
1. *Reduce the 18:00–22:00 peak.* Introduce time-of-use pricing and demand-response programmes that move flexible loads out of the evening window.
2. *Prepare for July–August, especially Zone 3.* Plan reserve capacity, run cooling-efficiency campaigns, and use temperature forecasts (for example, days above 30°C) to trigger early alerts.
3. *Schedule maintenance in low-demand windows.* Use overnight hours (about 03:00–07:00), Sundays and the low-demand months (November–December).
4. *Tailor programmes by zone.* Treat Zone 2's winter demand and Zone 3's summer spikes as separate problems.
5. *Improve the data.* Check the wind sensor, add several years of data and holiday information, and then build a demand forecasting model.

### Limitations
This is one year of data from 2017. The results show associations, not causes, and the units and zone descriptions are not documented.